# Введение в MapReduce модель на Python


In [62]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [63]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [64]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [65]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [66]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [67]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [68]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [69]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [70]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [71]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [72]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [73]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [74]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [75]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [76]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.601904258138054)),
 (1, np.float64(1.601904258138054)),
 (2, np.float64(1.601904258138054)),
 (3, np.float64(1.601904258138054)),
 (4, np.float64(1.601904258138054))]

## Inverted index

In [77]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [78]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [79]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [80]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('is', 18), ('it', 18), ('what', 10)]),
 (1, [('a', 2), ('banana', 2)])]

## TeraSort

In [81]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.01885287852217188)),
   (None, np.float64(0.03580452186431626)),
   (None, np.float64(0.07355292833133897)),
   (None, np.float64(0.12604331380017686)),
   (None, np.float64(0.132404508323276)),
   (None, np.float64(0.13513089946364076)),
   (None, np.float64(0.14890179130897918)),
   (None, np.float64(0.212168415754181)),
   (None, np.float64(0.21335333550108726)),
   (None, np.float64(0.254234361023323)),
   (None, np.float64(0.27773127952851107)),
   (None, np.float64(0.30385257904080976)),
   (None, np.float64(0.31699984105221723)),
   (None, np.float64(0.37425987841112707)),
   (None, np.float64(0.3804368006863472)),
   (None, np.float64(0.42506099929194285))]),
 (1,
  [(None, np.float64(0.518203268312839)),
   (None, np.float64(0.5948450828417855)),
   (None, np.float64(0.6342051326370997)),
   (None, np.float64(0.6357476841886947)),
   (None, np.float64(0.642474249569424)),
   (None, np.float64(0.6440688651318937)),
   (None, np.float64(0.698029580218

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [82]:
numbers = [3, 17, 2, 99, 45, 12]

def RECORDREADER():
    for i, value in enumerate(numbers):
        yield (i, value)

def MAP(_, value:int):
    yield ("max", value)

def REDUCE(key:str, values:Iterator[int]):
    current_max = None
    for v in values:
        if current_max is None or v > current_max:
            current_max = v
    yield (key, current_max)


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('max', 99)]

### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [83]:
def RECORDREADER():
    for i, value in enumerate(numbers):
        yield (i, value)

def MAP(_, value:float):
    yield ("avg", (value, 1))

def REDUCE(key:str, values:Iterator[Tuple[float,int]]):
    total_sum = 0.0
    total_count = 0

    for s, c in values:
        total_sum += s
        total_count += c

    yield (key, total_sum / total_count if total_count > 0 else 0)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('avg', 29.666666666666668)]

### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [84]:
def groupbykey_sort(iterable):
    sorted_data = sorted(iterable, key=lambda x: x[0])

    current_key = None
    current_values = []

    for k, v in sorted_data:
        if k != current_key:
            if current_key is not None:
                yield (current_key, current_values)
            current_key = k
            current_values = [v]
        else:
            current_values.append(v)

    if current_key is not None:
        yield (current_key, current_values)

data = [("a",1),("b",2),("a",3),("b",4),("c",5)]
print(list(groupbykey_sort(data)))

[('a', [1, 3]), ('b', [2, 4]), ('c', [5])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [85]:
data = np.random.randint(0, 20, 50)

maps = 3
reducers = 2

def INPUTFORMAT():
    split_size = int(np.ceil(len(data)/maps))

    def RECORDREADER(split):
        for value in split:
            yield (None, value)

    for i in range(0, len(data), split_size):
        yield RECORDREADER(data[i:i+split_size])

def MAP(_, value):
    yield (value, None)

def COMBINER(value, _):
    yield (value, None)

def REDUCE(value, _):
    yield (value, None)

partitioned_output = MapReduceDistributed(
    INPUTFORMAT,
    MAP,
    REDUCE,
    COMBINER=COMBINER
)

result = []
for (_, partition) in partitioned_output:
    result.extend([k for (k,_) in partition])

print(sorted(result))

36 key-value pairs were sent over a network.
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [86]:
R = [
    (1,"Alice",25),
    (2,"Bob",17),
    (3,"Carol",30),
    (4,"Dave",15)
]

def RECORDREADER():
    for i, tuple_ in enumerate(R):
        yield (i, tuple_)

def MAP(_, t):
    if t[2] >= 18:
        yield (t, t)

def REDUCE(t, values):
    for v in values:
        yield (t, v)

output = MapReduce(RECORDREADER, MAP, REDUCE)

print([v for (_,v) in output])

[(1, 'Alice', 25), (3, 'Carol', 30)]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [87]:
R = [
    (1, "Alice", 25),
    (2, "Bob", 17),
    (3, "Carol", 30),
    (4, "Alice", 25)
]

def project(t):
    return (t[1], t[2])

def RECORDREADER():
    for i, t in enumerate(R):
        yield (i, t)

def MAP(_, t):
    t_prime = project(t)
    yield (t_prime, t_prime)

def REDUCE(t_prime, values):
    yield (t_prime, t_prime)

output = MapReduce(RECORDREADER, MAP, REDUCE)
print([v for (_, v) in output])

[('Alice', 25), ('Bob', 17), ('Carol', 30)]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [88]:
R = [(1,"Alice"), (2,"Bob"), (3,"Carol")]
S = [(3,"Carol"), (4,"Dave")]

def RECORDREADER():
    for t in R:
        yield ("R", t)
    for t in S:
        yield ("S", t)

def MAP(_, t):
    yield (t, t)

def REDUCE(t, values):
    yield (t, t)

output = MapReduce(RECORDREADER, MAP, REDUCE)
print([v for (_, v) in output])

[(1, 'Alice'), (2, 'Bob'), (3, 'Carol'), (4, 'Dave')]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [89]:
R = [(1,"Alice"), (2,"Bob"), (3,"Carol")]
S = [(3,"Carol"), (4,"Dave")]

def RECORDREADER():
    for t in R:
        yield ("R", t)
    for t in S:
        yield ("S", t)

def MAP(_, t):
    yield (t, t)

def REDUCE(t, values):
    vals = list(values)
    if len(vals) == 2:
        yield (t, t)

output = MapReduce(RECORDREADER, MAP, REDUCE)
print([v for (_, v) in output])

[(3, 'Carol')]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [90]:
R = [(1,"Alice"), (2,"Bob"), (3,"Carol")]
S = [(3,"Carol"), (4,"Dave")]

def RECORDREADER():
    for t in R:
        yield ("R", t)
    for t in S:
        yield ("S", t)

def MAP(source, t):
    yield (t, source)

def REDUCE(t, sources):
    srcs = list(sources)
    if srcs == ["R"]:
        yield (t, t)

output = MapReduce(RECORDREADER, MAP, REDUCE)
print([v for (_, v) in output])

[(1, 'Alice'), (2, 'Bob')]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [91]:
R = [(1,"x"), (2,"y"), (3,"z")]
S = [("x",100), ("z",200), ("w",300)]

def RECORDREADER():
    for (a,b) in R:
        yield ("R", (a,b))
    for (b,c) in S:
        yield ("S", (b,c))

def MAP(source, t):
    if source == "R":
        a,b = t
        yield (b, ("R", a))
    else:
        b,c = t
        yield (b, ("S", c))

def REDUCE(b, values):
    Rs = []
    Ss = []

    for tag, val in values:
        if tag == "R":
            Rs.append(val)
        else:
            Ss.append(val)

    for a in Rs:
        for c in Ss:
            yield (None, (a,b,c))

output = MapReduce(RECORDREADER, MAP, REDUCE)
print([v for (_, v) in output])

[(1, 'x', 100), (3, 'z', 200)]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [92]:
R = [
    ("A", 10, 1),
    ("A", 20, 2),
    ("B", 5, 3),
    ("B", 15, 4),
    ("C", 7, 5)
]

def RECORDREADER():
    for i, t in enumerate(R):
        yield (i, t)

def MAP(_, t):
    a, b, c = t
    yield (a, b)

from typing import Iterator

def REDUCE(a, values:Iterator[int]):
    total = 0
    for v in values:
        total += v
    yield (a, total)

output = MapReduce(RECORDREADER, MAP, REDUCE)
print(list(output))

[('A', 30), ('B', 20), ('C', 7)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [93]:
A = [
    (0,0,1),(0,1,2),
    (1,0,3),(1,1,4)
]

x = [
    (0,10),
    (1,20)
]

def RECORDREADER():
    for (i,j,v) in A:
        yield ("A", (i,j,v))
    for (j,v) in x:
        yield ("X", (j,v))

def MAP(source, t):
    if source == "A":
        i,j,v = t
        yield (j, ("A", i, v))
    else:
        j,v = t
        yield (j, ("X", v))

def REDUCE(j, values):
    A_vals = []
    X_vals = []

    for val in values:
        if val[0] == "A":
            _, i, a_val = val
            A_vals.append((i, a_val))
        else:
            _, x_val = val
            X_vals.append(x_val)

    for (i, a_val) in A_vals:
        for x_val in X_vals:
            yield (i, a_val * x_val)

intermediate = list(MapReduce(RECORDREADER, MAP, REDUCE))

def RECORDREADER2():
    for (i,val) in intermediate:
        yield (i,val)

def MAP2(i,val):
    yield (i,val)

def REDUCE2(i, values):
    yield (i, sum(values))

output = MapReduce(RECORDREADER2, MAP2, REDUCE2)

print(list(output))

[(0, 50), (1, 110)]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [94]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [95]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
    (j, k) = k1
    w = v1
    for i in range(small_mat.shape[0]):
        yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
    total = 0.0
    for v in values:
        total += v
    yield (key, total)

Проверьте своё решение

In [96]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [97]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [98]:
A = [
    (0,0,1),(0,1,2),(0,2,3),
    (1,0,4),(1,1,5),(1,2,6)
]

B = [
    (0,0,7),(0,1,8),
    (1,0,9),(1,1,10),
    (2,0,11),(2,1,12)
]

def RECORDREADER():
    for (i,k,v) in A:
        yield ("A", (i,k,v))
    for (k,j,v) in B:
        yield ("B", (k,j,v))

def MAP(source, t):
    if source == "A":
        i,k,v = t
        yield (k, ("A", i, v))
    else:
        k,j,v = t
        yield (k, ("B", j, v))

def REDUCE(k, values):
    A_vals = []
    B_vals = []

    for val in values:
        if val[0] == "A":
            _, i, a_val = val
            A_vals.append((i, a_val))
        else:
            _, j, b_val = val
            B_vals.append((j, b_val))

    for (i,a_val) in A_vals:
        for (j,b_val) in B_vals:
            yield ((i,j), a_val*b_val)


intermediate = list(MapReduce(RECORDREADER, MAP, REDUCE))

def RECORDREADER2():
    for (key,val) in intermediate:
        yield (key,val)

def MAP2(key,val):
    yield (key,val)

def REDUCE2(key, values):
    yield (key, sum(values))

output = list(MapReduce(RECORDREADER2, MAP2, REDUCE2))
print(sorted(output))

[((0, 0), 58), ((0, 1), 64), ((1, 0), 139), ((1, 1), 154)]


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [99]:

A = [
    (0,0,1),(0,1,2),(0,2,3),
    (1,0,4),(1,1,5),(1,2,6)
]

B = [
    (0,0,7),(0,1,8),
    (1,0,9),(1,1,10),
    (2,0,11),(2,1,12)
]

maps = 2
reducers = 2

def INPUTFORMAT():
    def RECORDREADER_A():
        for t in A:
            yield ("A", t)
    def RECORDREADER_B():
        for t in B:
            yield ("B", t)
    yield RECORDREADER_A()
    yield RECORDREADER_B()

def MAP(source, t):
    if source == "A":
        i,k,v = t
        yield (k, ("A", i, v))
    else:
        k,j,v = t
        yield (k, ("B", j, v))

def REDUCE(k, values):
    A_vals = []
    B_vals = []
    for val in values:
        if val[0] == "A":
            _, i, a_val = val
            A_vals.append((i, a_val))
        else:
            _, j, b_val = val
            B_vals.append((j, b_val))

    for (i,a_val) in A_vals:
        for (j,b_val) in B_vals:
            yield ((i,j), a_val*b_val)

stage1 = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

intermediate = []
for (_, partition) in stage1:
    intermediate.extend(list(partition))

def INPUTFORMAT2():
    def RR():
        for (key,val) in intermediate:
            yield (key,val)
    yield RR()

def MAP2(key,val):
    yield (key,val)

def REDUCE2(key, values):
    yield (key, sum(values))

stage2 = MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2)
result = []
for (_, partition) in stage2:
    result.extend(list(partition))

print(sorted(result))

12 key-value pairs were sent over a network.
12 key-value pairs were sent over a network.
[((0, 0), 58), ((0, 1), 64), ((1, 0), 139), ((1, 1), 154)]


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [100]:
A = [
    (0,0,1),(0,1,2),(0,2,3),
    (1,0,4),(1,1,5),(1,2,6)
]

B = [
    (0,0,7),(0,1,8),
    (1,0,9),(1,1,10),
    (2,0,11),(2,1,12)
]

maps = 4
reducers = 2

def INPUTFORMAT():
    splitA = np.array_split(A, 2)
    splitB = np.array_split(B, 2)

    for chunk in splitA:
        def RR_A(chunk=chunk):
            for t in chunk:
                yield ("A", tuple(t))
        yield RR_A()

    for chunk in splitB:
        def RR_B(chunk=chunk):
            for t in chunk:
                yield ("B", tuple(t))
        yield RR_B()

def MAP(source, t):
    if source == "A":
        i,k,v = t
        yield (k, ("A", i, v))
    else:
        k,j,v = t
        yield (k, ("B", j, v))

def REDUCE(k, values):
    A_vals = []
    B_vals = []
    for val in values:
        if val[0] == "A":
            _, i, a_val = val
            A_vals.append((i, a_val))
        else:
            _, j, b_val = val
            B_vals.append((j, b_val))

    for (i,a_val) in A_vals:
        for (j,b_val) in B_vals:
            yield ((i,j), a_val*b_val)

stage1 = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)

intermediate = []
for (_, partition) in stage1:
    intermediate.extend(list(partition))

def INPUTFORMAT2():
    def RR():
        for (key,val) in intermediate:
            yield (key,val)
    yield RR()

def MAP2(key,val):
    yield (key,val)

def REDUCE2(key, values):
    yield (key, sum(values))

stage2 = MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2)

result = []
for (_, partition) in stage2:
    result.extend(list(partition))

print(sorted(result))

12 key-value pairs were sent over a network.
12 key-value pairs were sent over a network.
[((np.int64(0), np.int64(0)), np.int64(58)), ((np.int64(0), np.int64(1)), np.int64(64)), ((np.int64(1), np.int64(0)), np.int64(139)), ((np.int64(1), np.int64(1)), np.int64(154))]
